### 1. colab 연동

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!cp /content/drive/MyDrive/cifar100/cifar100.zip /content/
!unzip -q /content/cifar100.zip -d /content/dataset/

from sklearn.model_selection import train_test_split
from torchsummary import summary
from tqdm import tqdm
import torch.optim.lr_scheduler as lr_scheduler

### 2. CIFAR100 data로 train, test dataset,loader 만들기

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

train_dir = 'dataset/cifar100/train'
test_dir = 'dataset/cifar100/test'

transform = transforms.Compose([
    # transforms.Resize((224, 224)),
    transforms.ToTensor(),
    # transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    ]
)

train_dataset = datasets.ImageFolder(root=train_dir, transform=transform)
test_dataset = datasets.ImageFolder(root=test_dir, transform=transform)

train_loader_128 = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader_128 = DataLoader(test_dataset, batch_size=128, shuffle=False)

print(f'Number of training samples: {len(train_dataset)}')
print(f'Number of testing samples: {len(test_dataset)}')
print(f'Number of classes: {len(train_dataset.classes)}')
print(f'Class names: {train_dataset.classes}')
print(f'Example image shape: {train_dataset[0][0].shape}')

### 3. 모델 정의


In [ ]:
import torch.nn as nn

class depthwise_separable_conv(nn.Module):
    def __init__(self,in_channels,out_channels,stride, activation=nn.ReLU, norm_layer=nn.BatchNorm2d, layer_suffle=False):
        super(depthwise_separable_conv,self).__init__()
        if layer_suffle:
            self.dconv = nn.Sequential(
                nn.Conv2d(in_channels,in_channels,3,stride,1,groups=in_channels),
                activation(),
                norm_layer(in_channels)
            )
            self.conv = nn.Sequential(
                nn.Conv2d(in_channels,out_channels,1,1),
                activation(),
                norm_layer(out_channels)
            )
        else:
            self.dconv = nn.Sequential(
                nn.Conv2d(in_channels,in_channels,3,stride,1,groups=in_channels),
                norm_layer(in_channels),
                activation()
            )
            self.conv = nn.Sequential(
                nn.Conv2d(in_channels,out_channels,1,1),
                norm_layer(out_channels),
                activation()
            )

    def forward(self,x):
        out = self.dconv(x)
        out = self.conv(out)

        return out


class MobileNet(nn.Module):
    def __init__(self,a=1, activation=nn.ReLU, norm_layer=nn.BatchNorm2d, layer_suffle=False, num_layers=8):
        super(MobileNet,self).__init__()
        set_activation = activation
        set_norm_layer = norm_layer
        set_layer_suffle = layer_suffle

        self.conv1 = nn.Sequential(
            nn.Conv2d(3,32*a,3,2,1),
            norm_layer(32*a),
            activation()
        )

        self.Mobile = nn.Sequential(
            depthwise_separable_conv(32*a,64,1, set_activation, set_norm_layer, set_layer_suffle),
            depthwise_separable_conv(64,128,2, set_activation, set_norm_layer, set_layer_suffle),
            depthwise_separable_conv(128,128,1, set_activation, set_norm_layer, set_layer_suffle),
            depthwise_separable_conv(128,256,2, set_activation, set_norm_layer, set_layer_suffle),
            depthwise_separable_conv(256,256,1, set_activation, set_norm_layer, set_layer_suffle),
            depthwise_separable_conv(256,512,2, set_activation, set_norm_layer, set_layer_suffle),
            depthwise_separable_conv(512,1024,1, set_activation, set_norm_layer, set_layer_suffle),
            nn.AdaptiveAvgPool2d(1)
        )

        self.Mobile = nn.Sequential(*self.Mobile[:(-1-(8-num_layers))], self.Mobile[-1])
        self.final_channels = [32, 64, 128, 128, 256, 256, 512, 1024]

        self.FC = nn.Sequential(
            nn.Linear(self.final_channels[num_layers-1],100)
        )

    def forward(self,x):
        out = self.conv1(x)
        out = self.Mobile(out)
        out = out.view(out.size(0),-1)
        out = self.FC(out)

        return out

In [ ]:
summary(MobileNet().to(device), (3,32,32))

### 4. train, test 함수 정의

In [ ]:
def train(dataloader , model , loss_fn , optimizer , lr_scheduler):
    size = 0
    num_batches = len(dataloader)

    model.train()
    epoch_loss , epoch_correct = 0 , 0

    for i ,(data_ , target_) in enumerate(dataloader):
        data_ , target_ = data_.to(device), target_.to(device)
        optimizer.zero_grad()

        output_ = model(data_)

        loss = loss_fn(output_, target_)
        loss.backward()
        optimizer.step()

        pred = output_.argmax(dim=1)
        correct = (pred == target_).sum().item()
        epoch_correct += correct
        epoch_loss += loss.item()
        size += len(data_)

    train_acc = epoch_correct/size
    lr_scheduler.step()

    return train_acc , epoch_loss / num_batches

In [ ]:
def test(dataloader , model , loss_fn):
    size = 0
    num_baches = len(dataloader)
    epoch_loss , epoch_correct= 0 ,0
    with torch.no_grad(): # grad 연산 X
        model.eval() # evaluation dropout 연산시
        for i, (data_ , target_) in enumerate(dataloader):

            data_ , target_ = data_.to(device), target_.to(device)
            output_ = model(data_)
            loss = loss_fn(output_, target_)

            pred = output_.argmax(dim=1)
            correct = (pred == target_).sum().item()
            epoch_correct += correct
            epoch_loss += loss.item()
            size += len(data_)

    test_acc = epoch_correct/size

    return test_acc  , epoch_loss / num_baches

### 5. 학습 및 테스트

In [ ]:
EPOCHS = 15
suffle_test_logs = {"Conv_Norm_ReLU_acc":[],
                    "Conv_ReLU_Norm_acc":[],
                    "Conv_LeakyReLU_Norm_acc":[]
                    }

gn_layer = lambda channels: nn.GroupNorm(num_groups=4, num_channels=channels)
models = {
    "Conv_Norm_ReLU": MobileNet(activation=nn.ReLU, num_layers=1).to(device),
    "Conv_ReLU_Norm": MobileNet(layer_suffle=True).to(device),
    "Conv_LeakyReLU_Norm": MobileNet(activation=nn.LeakyReLU, layer_suffle=True).to(device)
}
models_name = list(models.keys())
criterion = nn.CrossEntropyLoss()

In [ ]:
# activation별 모델 학습
suffle_test_logs_name = list(suffle_test_logs.keys())
iteration = 0
for iteration in range(len(models)):
    current_model = models[models_name[iteration]]
    optimizer = optim.SGD(current_model.parameters(), 1e-2, momentum=0.9, nesterov=True, weight_decay=5e-4)
    scheduler = lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    print('='*50)
    print(f'current_model: {models_name[iteration]}')
    print('='*50)
    for epoch in tqdm(range(EPOCHS)):
        train_acc, train_loss = train(train_loader_128, current_model, criterion, optimizer, scheduler)
        test_acc, test_loss = test(test_loader_128, current_model, criterion)

        print('\n'f'train_acc:{train_acc:.4f} test_acc:{test_acc:.4f}')
        
    suffle_test_logs[suffle_test_logs_name[iteration]].append(test_acc)

### 7. 시각화

In [ ]:
import matplotlib.pyplot as plt

# activation별 모델 정확도 시각화
plt.figure(figsize=(10, 6))
layers = range(len(suffle_test_logs["Conv_Norm_ReLU_acc"]))

plt.plot(layers, suffle_test_logs["Conv_Norm_ReLU_acc"], 'b-o', label='Conv_Norm_ReLU Accuracy')
plt.plot(layers, suffle_test_logs["Conv_ReLU_Norm_acc"], 'r-s', label='Conv_ReLU_Norm Accuracy')
plt.plot(layers, suffle_test_logs["Conv_LeakyReLU_Norm_acc"], 'g-^', label='Conv_LeakyReLU_Norm Accuracy')

plt.title(f'Model Accuracy by the order of layers', fontsize=15)
plt.xlabel('Layers', fontsize=12)
plt.ylabel('Test Accuracy', fontsize=12)
plt.legend()

plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

In [ ]:
# batchsize별 모델 정확도 시각화
plt.figure(figsize=(10, 6))
layers = range(1, 7)

plt.plot(layers, batchsize_test_logs["batch_norm_acc"], 'b-o', label='Batch Norm Accuracy')
plt.plot(layers, batchsize_test_logs["group_norm_acc"], 'r-s', label='Group Norm Accuracy')
plt.title(f'Model Accuracy by batch_size', fontsize=15)
plt.xlabel('Batch Size', fontsize=12)
plt.ylabel('Test Accuracy', fontsize=12)
plt.legend()

plt.grid(True, linestyle='--', alpha=0.6)
plt.show()